# Task-routed: within-phase screening **by task similarity** (Option A)

Loads `sim_*.npz` from **one or more** run folders, computes the same screening metrics as `nb_task_routed_within_phase_signal_test.ipynb`, and stratifies by **task similarity** parsed from participant IDs.

**Participant ID patterns** (see `a1b2.analysis.within_phase_screening`):
- Human batches: `study1_far_sub12`, `study2_near_sub3` → similarity = `far` / `near` / `same`
- Geometry phantoms: `geom_sub_same`, `geom_sub_far_1` → similarity from `geom_sub_*`

**Metrics:** B-phase (probe 1 = B-routed) hidden norm ratio, B loss/acc; A2 probe-0 vs probe-1 loss gap; `has_core_comms_keys`.

**Configure** `RUN_FOLDERS` or use auto-discovery below, then run all cells.


In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.analysis.within_phase_screening import (
    build_overview_long,
    discover_task_routed_folders,
    parse_task_similarity_from_participant,
    parse_study_cohort_from_participant,
)

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None

simulations_root = project_root / "data" / "simulations"
SIMILARITY_ORDER = ["same", "near", "far"]
COHORT_ORDER = ["study1", "study2", "geom", "unknown"]

plt.rcParams["figure.figsize"] = (10, 4)
print("Project root:", project_root)
print("Simulations root:", simulations_root)


## Configure run folders

- **`AUTO_DISCOVER_TASK_ROUTED`**: if `True`, scan `data/simulations/` for child folders whose name contains `task_routed` and contain `sim_*.npz`.
- Or set **`RUN_FOLDERS`** to a list of explicit `Path`s (strings OK).

Multiple folders typically correspond to different **hyperparameters** (e.g. sparsity); all rows get a `run_folder` column. Similarity always comes from **participant** id, not folder name.

**Performance:** scanning every `task_routed` folder and many `npz` files can take several minutes. For a quick check, set `AUTO_DISCOVER_TASK_ROUTED = False` and set `RUN_FOLDERS` to a **single** run directory.


In [ ]:
# --- edit ---
AUTO_DISCOVER_TASK_ROUTED = True
RUN_FOLDERS = [
    # simulations_root / "two_module_rnn_25_task_routed_no_comms_nb2_nb2_task_routed_sp0_sep_cr_RNN",
]
# ----------------

if AUTO_DISCOVER_TASK_ROUTED:
    RUN_FOLDERS = discover_task_routed_folders(simulations_root)
    print(f"Discovered {len(RUN_FOLDERS)} task_routed folders under simulations")
    for p in RUN_FOLDERS[:15]:
        print(" ", p.name)
    if len(RUN_FOLDERS) > 15:
        print(f"  ... and {len(RUN_FOLDERS) - 15} more")
else:
    RUN_FOLDERS = [Path(p) if not isinstance(p, Path) else p for p in RUN_FOLDERS]
    RUN_FOLDERS = [simulations_root / p if not p.is_absolute() else p for p in RUN_FOLDERS]

RUN_FOLDERS = [p for p in RUN_FOLDERS if p.is_dir()]
if not RUN_FOLDERS:
    raise FileNotFoundError("No run folders configured or discovered.")


## Build long table + QC

Rows with `task_similarity` is `NaN` could not be parsed — inspect and extend `parse_task_similarity_from_participant` if needed.


In [ ]:
overview_long = build_overview_long(RUN_FOLDERS)

if "error" in overview_long.columns:
    bad = overview_long[overview_long["error"].notna()]
    if len(bad):
        print("Load errors:")
        display(bad[["run_folder", "participant", "error"]].head(20))

plot_df = overview_long[overview_long["error"].isna()].copy() if "error" in overview_long.columns else overview_long.copy()
if len(plot_df) == 0:
    raise RuntimeError("No valid npz rows (empty run folders or all load errors).")
print(f"Total rows (ok): {len(plot_df)} from {plot_df['run_folder'].nunique()} run folders")

unmatched = plot_df["task_similarity"].isna()
if unmatched.any():
    print(f"Unmatched similarity (n={unmatched.sum()}), sample participants:")
    display(plot_df.loc[unmatched, "participant"].drop_duplicates().head(25))
else:
    print("All participants matched to same/near/far.")

plot_df = plot_df.copy()
plot_df["task_similarity"] = pd.Categorical(
    plot_df["task_similarity"], categories=SIMILARITY_ORDER, ordered=True
)
plot_df["study_cohort"] = pd.Categorical(
    plot_df["study_cohort"], categories=COHORT_ORDER, ordered=True
)

display(plot_df.head(10))


## Counts: similarity × cohort × run folder


In [ ]:
ct = (
    plot_df.groupby(["task_similarity", "study_cohort"], dropna=False)
    .size()
    .rename("n")
    .reset_index()
)
display(ct.pivot_table(index="task_similarity", columns="study_cohort", values="n", aggfunc="sum", fill_value=0))

print("Rows per run_folder:")
display(plot_df.groupby("run_folder").size().sort_values(ascending=False).head(25))


## Summary statistics by task similarity

Key columns for comms screening: **`b_median_hA_over_hB_probe1`**, **`a2_loss_gap_p1_minus_p0`**.


In [ ]:
metric_cols = [
    "b_median_hA_over_hB_probe1",
    "b_median_hA_over_hB_all",
    "b_mean_loss_probe1",
    "b_mean_acc_probe1",
    "a2_loss_gap_p1_minus_p0",
    "a2_mean_loss_probe0",
    "a2_mean_loss_probe1",
]
metric_cols = [c for c in metric_cols if c in plot_df.columns]

g = plot_df.dropna(subset=["task_similarity"]).groupby("task_similarity", observed=True)
display(g[metric_cols].agg(["count", "mean", "std", "median"]).T)


## Figures: distributions by similarity


In [ ]:
dfp = plot_df.dropna(subset=["task_similarity"]).copy()

def box_by_similarity(y, ax, title):
    if dfp[y].notna().sum() == 0:
        ax.set_title(f"{title}\n(no data)")
        return
    if sns is not None:
        sns.boxplot(
            data=dfp,
            x="task_similarity",
            y=y,
            order=SIMILARITY_ORDER,
            ax=ax,
            palette="Set2",
        )
    else:
        data = [dfp.loc[dfp["task_similarity"] == s, y].dropna().values for s in SIMILARITY_ORDER if (dfp["task_similarity"] == s).any()]
        labels = [s for s in SIMILARITY_ORDER if (dfp["task_similarity"] == s).any()]
        if data:
            ax.boxplot(data, labels=labels)
    ax.set_title(title)
    ax.set_xlabel("task similarity")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
box_by_similarity("b_median_hA_over_hB_probe1", axes[0, 0], "B phase, probe 1: median ‖h_A‖/‖h_B‖")
box_by_similarity("b_mean_loss_probe1", axes[0, 1], "B phase, probe 1: mean loss")
box_by_similarity("a2_loss_gap_p1_minus_p0", axes[1, 0], "A2: mean loss (probe1 − probe0)")
box_by_similarity("a2_mean_acc_probe1", axes[1, 1], "A2: mean acc (probe 1)")
plt.tight_layout()
plt.show()


### Optional: hue by study cohort (study1 vs study2 vs geom)

Only useful when multiple cohorts appear **within** the same similarity level.


In [ ]:
dfp2 = plot_df.dropna(subset=["task_similarity"]).copy()
if dfp2["study_cohort"].nunique() > 1 and sns is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.boxplot(
        data=dfp2,
        x="task_similarity",
        y="a2_loss_gap_p1_minus_p0",
        hue="study_cohort",
        order=SIMILARITY_ORDER,
        hue_order=[c for c in COHORT_ORDER if c in set(dfp2["study_cohort"].astype(str))],
        ax=ax,
    )
    ax.set_title("A2 loss gap by similarity and cohort")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("Skip cohort hue: single cohort or seaborn missing.")


## Save table for other notebooks / figures


In [ ]:
out_csv = project_root / "data" / "derived" / "within_phase_overview_by_similarity.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
overview_long.to_csv(out_csv, index=False)
print("Wrote:", out_csv)


## Interpretation notes

- **Similarity** is read from the **participant** string, not the run folder. If one run only contains `study1_far_sub*`, all rows will be `far` — compare **across run folders** (e.g. sparsity) with `run_folder` as a second factor.
- **`b_median_hA_over_hB_probe1`**: use **probe 1** for B-routed input; mixed-probe "all" metrics are misleading when both probes appear in the B block.
- **No shift by similarity** may mean the metric is weak, power is low, or geometry mainly affects outcomes you have not merged yet (e.g. transfer from the interference notebook).
- **`has_core_comms_keys`**: if all `False`, regenerate sims with post-phase core/comms saving for pathway-level follow-up.
